## Ran Uram 206661886 
## Shahar Lankry 322659137 
## Daniel Geron 212515522
-------------------------

## Imports

In [92]:
import warnings
warnings.filterwarnings('ignore')
import cv2
import numpy as np
import os
from pathlib import Path
import argparse
import pandas as pd
import re
from typing import Tuple
from tqdm import tqdm
from openpyxl import load_workbook

## Change paths to match your folders location

In [93]:
og_images_folder=r"C:/Users/ranor/Desktop/ocr/Handwriting-OCR-for-Graphology/Data"
normalised_images_folder=r"C:/Users/ranor/Desktop/ocr/Handwriting-OCR-for-Graphology/Data_normalized"
feature_tables_location=r"C:/Users/ranor/Desktop/ocr/Handwriting-OCR-for-Graphology/tables"

#creating folders for results
visualization_folder=r"C:/Users/ranor/Desktop/ocr/Handwriting-OCR-for-Graphology/visualization_results"
if not os.path.exists(visualization_folder):
    os.makedirs(visualization_folder)
for feature in ['baseline','slant','stroke_thickness','right_margin','left_margin','word_aspect_ratio','baseline_slope']:
    feature_folder=os.path.join(visualization_folder,feature)
    if not os.path.exists(feature_folder):
        os.makedirs(feature_folder)

## Image normalization

In [94]:
def binarize_image(img: np.ndarray, method: str = 'sauvola') -> np.ndarray:
    # Converts grayscale image to binary (black and white) using thresholding.
    # This separates the ink (text) from the paper (background).

    if method == 'otsu':
        # Otsu finds a single global threshold for the whole image (good for uniform lighting)
        _, binary = cv2.threshold(img, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    elif method == 'adaptive':
        # Adaptive calculates threshold locally for small regions (better for shadows/uneven lighting)
        binary = cv2.adaptiveThreshold(
            img, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
            cv2.THRESH_BINARY, 21, 10
        )

    elif method == 'sauvola':
        # Sauvola is a specific adaptive method optimized for documents.
        # Here we use a tuned Adaptive Threshold as a robust fallback implementation.
        binary = cv2.adaptiveThreshold(
            img, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
            cv2.THRESH_BINARY, 25, 15
        )

    else:
        raise ValueError(f"Unknown binarization method: {method}")

    return binary


In [95]:
def remove_noise(binary_img: np.ndarray, min_component_size: int = 10) -> np.ndarray:
    # Cleans the image by removing small specks and dots that aren't part of the text.
    
    # Morphological Opening: Erodes then Dilates to remove tiny noise (salt-and-pepper noise)
    kernel = np.ones((2, 2), np.uint8)
    opened = cv2.morphologyEx(binary_img, cv2.MORPH_OPEN, kernel, iterations=1)

    # Connected Components: Identifies all separated blobs in the image
    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(255 - opened, connectivity=8)
    # Create a clean black canvas
    output = np.zeros_like(opened)

    # Copy only the components that are large enough (actual letters) to the new image
    for i in range(1, num_labels):
        area = stats[i, cv2.CC_STAT_AREA]
        if area >= min_component_size:
            output[labels == i] = 255

    # Invert back to ensure text is black on white background
    output = 255 - output

    return output


In [96]:
def crop_to_content(img: np.ndarray, padding: int = 20) -> np.ndarray:
    # Cuts away the empty white margins, keeping only the area with actual handwriting.
    
    # Find coordinates of all black pixels (text)
    coords = cv2.findNonZero(255 - img)

    if coords is None:
        return img

    # Get the bounding box (rectangle) that surrounds all the text
    x, y, w, h = cv2.boundingRect(coords)

    # Add some padding (white space) around the text so it doesn't touch the edges
    x = max(0, x - padding)
    y = max(0, y - padding)
    w = min(img.shape[1] - x, w + 2 * padding)
    h = min(img.shape[0] - y, h + 2 * padding)

    # Perform the actual crop
    cropped = img[y:y+h, x:x+w]

    return cropped

In [97]:
def normalize_height(img: np.ndarray, target_height: int = 64) -> np.ndarray:
    # Resizes the image to a fixed height (e.g., 64px) while keeping the Aspect Ratio.
    # This ensures the letters don't get squashed or stretched unnaturally.
    
    h, w = img.shape[:2]

    # Calculate new width based on aspect ratio
    aspect_ratio = w / h
    new_width = int(target_height * aspect_ratio)
    
    # Resize using INTER_AREA interpolation (best for shrinking images)
    resized = cv2.resize(img, (new_width, target_height), interpolation=cv2.INTER_AREA)

    return resized

In [98]:
def process_single_image(
    input_path: str, output_path: str, binarize: bool = True,
    denoise: bool = True, crop: bool = True, normalize_size: bool = False,
    target_height: int = 64, binarization_method: str = 'adaptive', min_noise_size: int = 10
) -> bool:
    # The main pipeline for a single file. Runs all steps in order.
    
    try:
        img = cv2.imread(input_path)
        if img is None:
            return False

        # Convert to grayscale first
        if len(img.shape) == 3:
            gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        else:
            gray = img

        processed = gray.copy()

        # Step 1: Binarization (Thresholding)
        if binarize:
            processed = binarize_image(processed, method=binarization_method)

        # Step 2: Noise Removal (Cleaning)
        if denoise:
            processed = remove_noise(processed, min_component_size=min_noise_size)

        # Step 3: Smart Cropping
        if crop:
            processed = crop_to_content(processed, padding=20)

        # Step 4: Height Normalization (Optional)
        if normalize_size and target_height > 0:
            processed = normalize_height(processed, target_height=target_height)

        # Save the final processed image
        cv2.imwrite(output_path, processed)

        return True

    except Exception as e:
        print(f"Error: {e}")
        return False


In [99]:
def process_directory(input_dir: str, output_dir: str, **kwargs) -> None:
    # Iterates over the entire folder and processes every image found.
    
    os.makedirs(output_dir, exist_ok=True)

    image_files = []
    # Find all image files with common extensions
    for ext in ['.png', '.jpg', '.jpeg', '.tif', '.tiff']:
        image_files.extend(Path(input_dir).glob(f'*{ext}'))
        image_files.extend(Path(input_dir).glob(f'*{ext.upper()}'))

    # Sort and remove duplicates
    image_files = sorted(list(set(image_files)))
    total = len(image_files)

    print(f"Found {total} images to process")
    print(f"Output directory: {output_dir}")
    print("-" * 60)

    success_count = 0

    # Loop through all images
    for idx, img_path in enumerate(image_files, 1):
        output_path = os.path.join(output_dir, img_path.name)

        # Process the current image
        success = process_single_image(str(img_path), output_path, **kwargs)

        if success:
            success_count += 1
            if idx % 100 == 0:
                print(f"Processed {idx}/{total} images...")
        else:
            print(f"[ERROR] Failed to process {img_path.name}")

    print("-" * 60)
    print(f"Done! Successfully processed: {success_count}/{total}")

In [100]:

    # 3. Processing Settings (Change to False if you want to skip a step)
should_binarize = True
should_denoise = True
should_crop = True
should_normalize_height = False
    

    # Check if input directory exists before starting
if 'og_images_folder' not in locals():
    print("Error: Variables not defined. Please run the cell with the paths first (Cell 2).")
elif not os.path.exists(og_images_folder):
    print(f"Error: The input folder does not exist at:\n{og_images_folder}")
else:
    print(f"Folder found! Processing images...")

    # Run the processing
    process_directory(
            input_dir=og_images_folder,
            output_dir=normalised_images_folder,
            binarize=should_binarize,
            denoise=should_denoise,
            crop=should_crop,
            normalize_size=should_normalize_height,
            target_height=64 
    )


Folder found! Processing images...
Found 2512 images to process
Output directory: C:/Users/ranor/Desktop/ocr/Handwriting-OCR-for-Graphology/Data_normalized
------------------------------------------------------------
Processed 100/2512 images...
Processed 200/2512 images...
Processed 300/2512 images...
Processed 400/2512 images...
Processed 500/2512 images...
Processed 600/2512 images...
Processed 700/2512 images...
Processed 800/2512 images...
Processed 900/2512 images...
Processed 1000/2512 images...
Processed 1100/2512 images...
Processed 1200/2512 images...
Processed 1300/2512 images...
Processed 1400/2512 images...
Processed 1500/2512 images...
Processed 1600/2512 images...
Processed 1700/2512 images...
Processed 1800/2512 images...
Processed 1900/2512 images...
Processed 2000/2512 images...
Processed 2100/2512 images...
Processed 2200/2512 images...
Processed 2300/2512 images...
Processed 2400/2512 images...
Processed 2500/2512 images...
------------------------------------------

## FEATURE EXTRACTION

<table dir="rtl" style="border-collapse: collapse; width: 100%; text-align: right; border: 1px solid black;">
    <thead>
        <tr>
            <th style="border: 1px solid black; padding: 10px;">ערכי הקיצון והאמצע (סקאלה 0-1)</th>
            <th style="border: 1px solid black; padding: 10px;">מה זה אומר? (תיאור)</th>
            <th style="border: 1px solid black; padding: 10px;">שם הפיצ'ר</th>
        </tr>
    </thead>
    <tbody>
        <tr>
            <td style="border: 1px solid black; padding: 10px;">0.0 = שמאלית חזקה (נגד הכיוון)<br>0.5 = אנכי לגמרי (90 מעלות, ישר)<br>1.0 = ימנית חזקה (שוכב קדימה)</td>
            <td style="border: 1px solid black; padding: 10px;">זווית הכתיבה ביחס לאנך</td>
            <td style="border: 1px solid black; padding: 10px;">נטייה (Slant)</td>
        </tr>
        <tr>
            <td style="border: 1px solid black; padding: 10px;">0.0 = דקיק (קו נימי, עדין מאוד)<br>0.5 = בינוני (עובי סטנדרטי)<br>1.0 = עבה מאוד (קו "בצקי", מרוח)</td>
            <td style="border: 1px solid black; padding: 10px;">עובי הקו ביחס לגודל האות ("לחץ")</td>
            <td style="border: 1px solid black; padding: 10px;">עובי קו (Stroke Thickness)</td>
        </tr>
        <tr>
            <td style="border: 1px solid black; padding: 10px;">0.0 = חותך/יורד (הכתיבה יורדת מתחת לקו)<br>0.5 = על הקו בדיוק (התאמה מושלמת)<br>1.0 = מרחף גבוה (הכתיבה מנותקת מהקו כלפי מעלה)</td>
            <td style="border: 1px solid black; padding: 10px;">מיקום האות ביחס לפס המודפס</td>
            <td style="border: 1px solid black; padding: 10px;">היצמדות לשורה (Baseline)</td>
        </tr>
        <tr>
            <td style="border: 1px solid black; padding: 10px;">0.0 = צמוד לקצה הימני (0% עד 5% מרוחב הדף)<br>0.5 = מרחק מאוזן מהקצה (10% עד 15%)<br>1.0 = מרחק גדול מהקצה הימני (מעל 30%)</td>
            <td style="border: 1px solid black; padding: 10px;">המרחק מהפיקסל השחור הראשון בשורה לקצה הימני של המסגרת</td>
            <td style="border: 1px solid black; padding: 10px;"><strong>מדידת שוליים- ימין (Margin Right)</strong></td>
        </tr>
        <tr>
            <td style="border: 1px solid black; padding: 10px;">0.0 = צמוד לקצה השמאלי (0% עד 5% מרוחב הדף)<br>0.5 = מרחק מאוזן מהקצה (10% עד 15%)<br>1.0 = מרחק גדול מהקצה השמאלי (מעל 30%)</td>
            <td style="border: 1px solid black; padding: 10px;">המרחק מהפיקסל האחרון לקצה השמאלי של המסגרת</td>
            <td style="border: 1px solid black; padding: 10px;"><strong>מדידת שוליים- שמאל (Margin Left)</strong></td>
        </tr>
        <tr>
            <td style="border: 1px solid black; padding: 10px;">-1 = זוהתה מילה אחת בלבד (לא ניתן למדוד)<br>0.0 = מילים דבוקות (אין רווח)<br>0.5 = רווח תקין בין מילים<br>1.0 = רווחים גדולים מאוד בין מילים</td>
            <td style="border: 1px solid black; padding: 10px;">המרחק החציוני בין מילים שזוהו בשורה</td>
            <td style="border: 1px solid black; padding: 10px;"><strong>רווח בין מילים (Word Spacing)</strong></td>
        </tr>
        <tr>
            <td style="border: 1px solid black; padding: 10px;">0.0 = כתב זעיר- תופס שטח מינימלי מהשורה<br>0.5 = כתב בינוני ותקני- פרופורציה הגיונית<br>1.0 = כתב ענק- משתלט על מרחב השורה</td>
            <td style="border: 1px solid black; padding: 10px;">המרחק האנכי של האותיות, מחושב כחציון גובה התיבות התוחמות ומנורמל לגובה הדף</td>
            <td style="border: 1px solid black; padding: 10px;"><strong>גודל הכתב האבסולוטי (Letter Size)</strong></td>
        </tr>
        <tr>
            <td style="border: 1px solid black; padding: 10px;">0.0 = כתב זוויתי- קווים חדים ושפיצים<br>0.5 = שילוב מאוזן- עקומות מתונות וזורמות<br>1.0 = כתב עגול- צורות עגולות, רכות ומלאות</td>
            <td style="border: 1px solid black; padding: 10px;">בדיקת ה"שפיציות" של הכתב דרך חישוב מדד המעגליות של החללים הלבנים בתוך האותיות</td>
            <td style="border: 1px solid black; padding: 10px;">מעגליות מול זוויתיות (Roundness vs. Angularity)</td>
        </tr>
        <tr>
            <td style="border: 1px solid black; padding: 10px;">0.0 = שורה נופלת (שיפוע רגרסיה חיובי)<br>0.5 = שורה ישרה (קו אופקי ושטוח)<br>1.0 = שורה מטפסת (שיפוע רגרסיה שלילי)</td>
            <td style="border: 1px solid black; padding: 10px;">כיוון הזרימה של השורה, מחושב באמצעות רגרסיה לינארית על מרכזי הכובד של המילים</td>
            <td style="border: 1px solid black; padding: 10px;">שיפוע קו הבסיס (Baseline Slope)</td>
        </tr>
        <tr>
            <td style="border: 1px solid black; padding: 10px;">0.0 = מילים צרות ומכווצות- דחוסות כלפי פנים<br>0.5 = פרופורציה מאוזנת- יחס רוחב-גובה תקין<br>1.0 = מילים רחבות ומתוחות- מרוחות על הדף</td>
            <td style="border: 1px solid black; padding: 10px;">בדיקת "מתיחת" המילה בחלל, מחושב כחציון היחס בין רוחב התיבה התוחמת לגובהה</td>
            <td style="border: 1px solid black; padding: 10px;">יחס רוחב-גובה של המילים (Word Aspect Ratio)</td>
        </tr>
    </tbody>
</table>

### Slant

### Stroke Thickness

### Baseline 

### Right Margin

### Left Margin

### Word Spacing

### Letter Size 

## Word Aspect Ratio

## Basline Slope

# Roundness vs. Angularity

## Unified Feature Table

Merge all per-feature Excel files into a single table — one row per image, one column per feature.

## Graphological Report

Send the unified feature table to Claude Opus and receive a full personal graphological report.

In [101]:
import cv2
import numpy as np
import pandas as pd
import os
from scipy.ndimage import uniform_filter1d

EXTRACTION_DIR = r"ransextraction"
TABLES_DIR     = os.path.join(EXTRACTION_DIR, "tables")
VIS_DIR        = os.path.join(EXTRACTION_DIR, "visualizations")

for d in [TABLES_DIR, VIS_DIR]:
    os.makedirs(d, exist_ok=True)

_COL_SPACING_MAX_RAW = 0.30
_MARGIN_MAX_RAW      = 0.30


def _label_on_image(vis, text, x=10, y=10):
    font, scale, thick = cv2.FONT_HERSHEY_SIMPLEX, 1.5, 2
    (tw, th), base = cv2.getTextSize(text, font, scale, thick)
    pad = 6
    cv2.rectangle(vis,
                  (x - pad, y - pad),
                  (x + tw + pad, y + th + base + pad),
                  (0, 0, 0), -1)
    cv2.putText(vis, text, (x, y + th),
                font, scale, (255, 255, 255), thick)


def _remove_junk(binary):
    """
    Remove structural blobs:
      1. Boundary-touching  -- outer frame, corner markers
      2. Large hollow       -- page frame when not touching boundary
         bw > 40% AND bh > 40% AND fill < 0.40
    """
    h, w = binary.shape
    n, labels, stats, _ = cv2.connectedComponentsWithStats(binary, connectivity=8)
    clean = binary.copy()
    for lbl in range(1, n):
        x_ = stats[lbl, cv2.CC_STAT_LEFT];  y_ = stats[lbl, cv2.CC_STAT_TOP]
        bw = stats[lbl, cv2.CC_STAT_WIDTH]; bh = stats[lbl, cv2.CC_STAT_HEIGHT]
        area = stats[lbl, cv2.CC_STAT_AREA]
        fill = area / (bw * bh + 1e-9)

        boundary    = (x_ == 0 or y_ == 0 or x_ + bw >= w or y_ + bh >= h)
        large_frame = (bw > w * 0.40 and bh > h * 0.40 and fill < 0.40)

        if boundary or large_frame:
            clean[labels == lbl] = 0
    return clean


def _binary(gray):
    _, b = cv2.threshold(gray, 127, 255, cv2.THRESH_BINARY_INV)
    return b


def calculate_column_spacing(img, filename="", save_vis=True):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) if len(img.shape) == 3 else img.copy()
    h, w = gray.shape
    clean = _remove_junk(_binary(gray))

    col_sums  = uniform_filter1d(clean.sum(axis=0).astype(float), size=max(1, w // 40))
    threshold = col_sums.max() * 0.05 if col_sums.max() > 0 else 1
    in_col    = col_sums > threshold

    bands, start = [], None
    for x, val in enumerate(in_col):
        if val and start is None:
            start = x
        elif not val and start is not None:
            bands.append([start, x])
            start = None
    if start is not None:
        bands.append([start, w])

    merged, min_gap = [], w // 50
    for b in bands:
        if merged and b[0] - merged[-1][1] < min_gap:
            merged[-1][1] = b[1]
        else:
            merged.append(b)

    if len(merged) < 2:
        return -1

    gaps  = [merged[i+1][0] - merged[i][1] for i in range(len(merged) - 1)]
    raw   = sum(gaps) / len(gaps) / w
    grade = round(min(1.0, raw / _COL_SPACING_MAX_RAW), 4)

    if save_vis and filename:
        vis = cv2.cvtColor(gray, cv2.COLOR_GRAY2BGR)
        for x1, x2 in merged:
            cv2.rectangle(vis, (x1, 0), (x2, h), (0, 200, 0), 2)
        for i in range(len(merged) - 1):
            gx1, gx2 = merged[i][1], merged[i+1][0]
            mid = h // 2
            cv2.rectangle(vis, (gx1, mid - 10), (gx2, mid + 10), (0, 0, 255), -1)
            cv2.putText(vis, f"{gaps[i]}px", (gx1 + 4, mid + 5),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
        _label_on_image(vis, f"columns_spacing: {grade:.4f}")
        cv2.imwrite(os.path.join(VIS_DIR, filename.replace(".png", "_col_spacing.png")), vis)

    return grade


def calculate_top_margin(img, filename="", save_vis=True):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) if len(img.shape) == 3 else img.copy()
    h, w = gray.shape
    clean = _remove_junk(_binary(gray))

    ink_rows = np.where(clean.sum(axis=1) > 0)[0]
    if len(ink_rows) == 0:
        return -1

    first_ink = int(ink_rows[0])
    grade     = round(min(1.0, (first_ink / h) / _MARGIN_MAX_RAW), 4)

    if save_vis and filename:
        vis     = cv2.cvtColor(gray, cv2.COLOR_GRAY2BGR)
        overlay = vis.copy()
        cv2.rectangle(overlay, (0, 0), (w, first_ink), (0, 200, 200), -1)
        vis = cv2.addWeighted(overlay, 0.35, vis, 0.65, 0)
        cv2.line(vis, (0, first_ink), (w, first_ink), (0, 0, 255), 2)
        _label_on_image(vis, f"top_margin: {grade:.4f}")
        cv2.imwrite(os.path.join(VIS_DIR, filename.replace(".png", "_top_margin.png")), vis)

    return grade


def calculate_bottom_margin(img, filename="", save_vis=True):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) if len(img.shape) == 3 else img.copy()
    h, w = gray.shape
    clean = _remove_junk(_binary(gray))

    ink_rows = np.where(clean.sum(axis=1) > 0)[0]
    if len(ink_rows) == 0:
        return -1

    last_ink = int(ink_rows[-1])
    grade    = round(min(1.0, ((h - 1 - last_ink) / h) / _MARGIN_MAX_RAW), 4)

    if save_vis and filename:
        vis     = cv2.cvtColor(gray, cv2.COLOR_GRAY2BGR)
        overlay = vis.copy()
        cv2.rectangle(overlay, (0, last_ink), (w, h), (0, 200, 200), -1)
        vis = cv2.addWeighted(overlay, 0.35, vis, 0.65, 0)
        cv2.line(vis, (0, last_ink), (w, last_ink), (0, 0, 255), 2)
        _label_on_image(vis, f"bottom_margin: {grade:.4f}")
        cv2.imwrite(os.path.join(VIS_DIR, filename.replace(".png", "_bottom_margin.png")), vis)

    return grade


In [102]:
import os
import glob
import cv2
import pandas as pd

INPUT_DIR      = r"cropped image to lines/attempt6"
FINAL_NORM_DIR = r"finalNormalization"
COLUMN_TOKEN   = "_COLUMNS"

os.makedirs(FINAL_NORM_DIR, exist_ok=True)

images = sorted(
    glob.glob(os.path.join(INPUT_DIR, "*.png")) +
    glob.glob(os.path.join(INPUT_DIR, "*.jpg"))
)

blank_records = []

for img_path in images:
    fname     = os.path.basename(img_path)
    norm_path = os.path.join(FINAL_NORM_DIR, fname)

    process_single_image(img_path, norm_path)
    norm_img = cv2.imread(norm_path)
    print(f"Normalized: {fname}")

    if COLUMN_TOKEN in fname:
        col_spacing   = calculate_column_spacing(norm_img, filename=fname)
        top_margin    = calculate_top_margin(norm_img, filename=fname)
        bottom_margin = calculate_bottom_margin(norm_img, filename=fname)
        print(f"  [blank] columns_spacing  = {col_spacing}")
        print(f"  [blank] top_margin       = {top_margin}")
        print(f"  [blank] bottom_margin    = {bottom_margin}")
        blank_records.append({
            "filename":        fname,
            "columns_spacing": col_spacing,
            "top_margin":      top_margin,
            "bottom_margin":   bottom_margin,
        })
    else:
        print("  [printed line] doing feature: slant")
        print("  [printed line] doing feature: stroke thickness")
        print("  [printed line] doing feature: baseline alignment")
        print("  [printed line] doing feature: word spacing")
        print("  [printed line] doing feature: letter size")
        print("  [printed line] doing feature: roundness vs angularity")
        print("  [printed line] doing feature: baseline slope")
        print("  [printed line] doing feature: word aspect ratio")

    print("  [shared] doing feature: left margin")
    print("  [shared] doing feature: right margin")
    print()

if blank_records:
    df = pd.DataFrame(blank_records)

    df[["filename", "columns_spacing"]].to_excel(
        os.path.join(TABLES_DIR, "columns_spacing.xlsx"), index=False)
    df[["filename", "top_margin"]].to_excel(
        os.path.join(TABLES_DIR, "top_margin.xlsx"), index=False)
    df[["filename", "bottom_margin"]].to_excel(
        os.path.join(TABLES_DIR, "bottom_margin.xlsx"), index=False)

    print(f"Saved columns_spacing.xlsx, top_margin.xlsx, bottom_margin.xlsx ({len(df)} rows each)")


Normalized: 4_6010333047398866303-1_COLUMNS.png
  [blank] columns_spacing  = 0.6899
  [blank] top_margin       = 0.2809
  [blank] bottom_margin    = 1.0
  [shared] doing feature: left margin
  [shared] doing feature: right margin

Normalized: 4_6010333047398866303-1_line_01.png
  [printed line] doing feature: slant
  [printed line] doing feature: stroke thickness
  [printed line] doing feature: baseline alignment
  [printed line] doing feature: word spacing
  [printed line] doing feature: letter size
  [printed line] doing feature: roundness vs angularity
  [printed line] doing feature: baseline slope
  [printed line] doing feature: word aspect ratio
  [shared] doing feature: left margin
  [shared] doing feature: right margin

Normalized: 4_6010333047398866303-1_line_02.png
  [printed line] doing feature: slant
  [printed line] doing feature: stroke thickness
  [printed line] doing feature: baseline alignment
  [printed line] doing feature: word spacing
  [printed line] doing feature: 